# Data Loading and Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Define the dataset path based on Kaggle environment
BASE_PATH = '/kaggle/input/playground-series-s6e4'

# =============================================================================
# STEP 1: DATA LOADING AND PREPROCESSING
# =============================================================================

print("Loading datasets...")
try:
    train_df = pd.read_csv(f'{BASE_PATH}/train.csv')
    test_df = pd.read_csv(f'{BASE_PATH}/test.csv')
    sample_sub = pd.read_csv(f'{BASE_PATH}/sample_submission.csv')
except FileNotFoundError:
    # Fallback path if the directory structure differs slightly
    FALLBACK_PATH = '/kaggle/input/competitions/playground-series-s6e4'
    train_df = pd.read_csv(f'{FALLBACK_PATH}/train.csv')
    test_df = pd.read_csv(f'{FALLBACK_PATH}/test.csv')
    sample_sub = pd.read_csv(f'{FALLBACK_PATH}/sample_submission.csv')

print(f"Train data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

# Separate Features (X) and Target (y)
print("Separating features and target...")
X = train_df.drop(columns=['id', 'Irrigation_Need'])
y_raw = train_df['Irrigation_Need']
X_test = test_df.drop(columns=['id'])

# Target Encoding
# Mapping Low=0, Medium=1, High=2
print("Encoding target variable...")
target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
y = y_raw.map(target_mapping)

# Categorical Feature Encoding
print("Encoding categorical features...")
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Use LabelEncoder for categorical variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Fit on combined train and test data to handle any unseen categories in test set
    combined_data = pd.concat([X[col], X_test[col]]).astype(str)
    le.fit(combined_data)
    
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

print("\n--- Step 1 Complete ---")
print(f"Final Features (X) shape: {X.shape}")
print(f"Final Test Features (X_test) shape: {X_test.shape}")
print(f"Target (y) value counts:\n{y.value_counts()}")

Loading datasets...
Train data shape: (630000, 21)
Test data shape: (270000, 20)
Separating features and target...
Encoding target variable...
Encoding categorical features...

--- Step 1 Complete ---
Final Features (X) shape: (630000, 19)
Final Test Features (X_test) shape: (270000, 19)
Target (y) value counts:
Irrigation_Need
0    369917
1    239074
2     21009
Name: count, dtype: int64


# Feature Engineering

In [2]:
# =============================================================================
# STEP 2: FEATURE ENGINEERING
# =============================================================================

def create_features(df):
    """
    Creates new interaction features based on agricultural logic to help 
    tree-based models capture complex patterns more easily.
    """
    df_engineered = df.copy()
    
    # 1. Dryness Stress Index
    # Ratio of temperature to soil moisture (adding a small epsilon to avoid division by zero)
    if 'Temperature_C' in df_engineered.columns and 'Soil_Moisture' in df_engineered.columns:
        df_engineered['Dryness_Stress_Index'] = df_engineered['Temperature_C'] / (df_engineered['Soil_Moisture'] + 1e-5)
        
    # 2. Climate Severity
    # Difference between temperature and precipitation
    if 'Temperature_C' in df_engineered.columns and 'Precipitation_mm' in df_engineered.columns:
        df_engineered['Temp_Minus_Precip'] = df_engineered['Temperature_C'] - df_engineered['Precipitation_mm']
        
    # 3. Water Retention Potential
    # Interaction between current soil moisture and recent precipitation
    if 'Soil_Moisture' in df_engineered.columns and 'Precipitation_mm' in df_engineered.columns:
        df_engineered['Moisture_Precip_Interaction'] = df_engineered['Soil_Moisture'] * df_engineered['Precipitation_mm']
        
    # 4. Crop & Mulch Interaction
    # Combining the previously label-encoded categorical features mathematically
    if 'Crop_Growth_Stage' in df_engineered.columns and 'Mulching_Used' in df_engineered.columns:
        df_engineered['Crop_Mulch_Interaction'] = (df_engineered['Crop_Growth_Stage'].astype(int) * 10) + df_engineered['Mulching_Used'].astype(int)
        
    return df_engineered

print("Applying feature engineering...")
X = create_features(X)
X_test = create_features(X_test)

print("\n--- Step 2 Complete ---")
print(f"Engineered Features (X) shape: {X.shape}")
print(f"Engineered Test Features (X_test) shape: {X_test.shape}")

# Optional: Display the newly created columns
original_cols = set(train_df.columns) - {'id', 'Irrigation_Need'}
new_cols = list(set(X.columns) - original_cols)
print(f"\nNew features created: {new_cols}")

Applying feature engineering...

--- Step 2 Complete ---
Engineered Features (X) shape: (630000, 21)
Engineered Test Features (X_test) shape: (270000, 21)

New features created: ['Dryness_Stress_Index', 'Crop_Mulch_Interaction']


# Cross-Validation Strategy (Stratified K-Fold)

In [3]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# =============================================================================
# STEP 3: CROSS-VALIDATION STRATEGY (STRATIFIED K-FOLD)
# =============================================================================

# Define the number of folds and random state for reproducibility
N_SPLITS = 5
RANDOM_STATE = 42

print(f"Setting up Stratified {N_SPLITS}-Fold Cross Validation...")

# Initialize StratifiedKFold
# This ensures each fold has the same proportion of target classes as the original dataset
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Prepare arrays to store Out-Of-Fold (OOF) predictions and test predictions
# These will hold the probabilities for the 3 classes: Low(0), Medium(1), High(2)
# and will be used later for ensemble/blending
oof_preds_lgb = np.zeros((len(X), 3))
test_preds_lgb = np.zeros((len(X_test), 3))

oof_preds_xgb = np.zeros((len(X), 3))
test_preds_xgb = np.zeros((len(X_test), 3))

oof_preds_cat = np.zeros((len(X), 3))
test_preds_cat = np.zeros((len(X_test), 3))

print("\n--- Step 3 Complete ---")
print(f"Cross-validation strategy initialized with {N_SPLITS} folds.")
print(f"OOF arrays prepared for LightGBM, XGBoost, and CatBoost (Shape: {oof_preds_lgb.shape}).")

Setting up Stratified 5-Fold Cross Validation...

--- Step 3 Complete ---
Cross-validation strategy initialized with 5 folds.
OOF arrays prepared for LightGBM, XGBoost, and CatBoost (Shape: (630000, 3)).


# Hyperparameter Tuning with Optuna

In [4]:
import optuna
import lightgbm as lgb
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
import numpy as np

# =============================================================================
# STEP 4: HYPERPARAMETER TUNING WITH OPTUNA (Example for LightGBM)
# =============================================================================

print("Starting Optuna Hyperparameter Tuning for LightGBM...")

def objective(trial):
    # Define the hyperparameter search space
    param = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_error',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'class_weight': 'balanced', # Crucial for imbalanced data
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
    }

    # Setup Stratified K-Fold inside the objective function
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # Using 3 folds for faster tuning
    cv_scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        # Train LightGBM model
        model = lgb.LGBMClassifier(**param)
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        # Predict and evaluate using Balanced Accuracy
        preds = model.predict(X_val_fold)
        score = balanced_accuracy_score(y_val_fold, preds)
        cv_scores.append(score)

    # Return the average Balanced Accuracy across folds
    return np.mean(cv_scores)

# Create a study object and optimize
# Direction is 'maximize' because we want the highest balanced accuracy
study = optuna.create_study(direction='maximize', study_name="LGBM_Tuning")
# Limit n_trials to a smaller number (e.g., 20-50) for testing, increase for final run
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n--- Step 4 Complete ---")
print("Best LightGBM Parameters found by Optuna:")
best_lgb_params = study.best_params
# Ensure essential fixed parameters are added back to the best params dictionary
best_lgb_params['objective'] = 'multiclass'
best_lgb_params['num_class'] = 3
best_lgb_params['class_weight'] = 'balanced'
best_lgb_params['random_state'] = 42

for key, value in best_lgb_params.items():
    print(f"  {key}: {value}")
print(f"Best CV Balanced Accuracy: {study.best_value:.4f}")

[I 2026-04-28 12:08:05,179] A new study created in memory with name: LGBM_Tuning


Starting Optuna Hyperparameter Tuning for LightGBM...


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-28 12:13:57,940] Trial 0 finished with value: 0.9678950074381083 and parameters: {'n_estimators': 666, 'learning_rate': 0.015130563221537642, 'max_depth': 6, 'num_leaves': 133, 'min_child_samples': 37, 'subsample': 0.882625166171522, 'colsample_bytree': 0.9493828808505997, 'reg_alpha': 0.0002905911788411268, 'reg_lambda': 1.2743001629466466e-07}. Best is trial 0 with value: 0.9678950074381083.
[I 2026-04-28 12:15:13,375] Trial 1 finished with value: 0.9704937641857247 and parameters: {'n_estimators': 217, 'learning_rate': 0.17471571596975127, 'max_depth': 4, 'num_leaves': 138, 'min_child_samples': 28, 'subsample': 0.8211430488867597, 'colsample_bytree': 0.5700367644474347, 'reg_alpha': 0.004124932749128112, 'reg_lambda': 0.007261742492105987}. Best is trial 1 with value: 0.9704937641857247.
[I 2026-04-28 12:20:08,352] Trial 2 finished with value: 0.9675517542421813 and parameters: {'n_estimators': 723, 'learning_rate': 0.07332110669203147, 'max_depth': 7, 'num_leaves': 32, '

# Model Training with Class Weights

In [5]:
import lightgbm as lgb
from sklearn.metrics import balanced_accuracy_score
import numpy as np

# =============================================================================
# STEP 5: MODEL TRAINING WITH CLASS WEIGHTS
# =============================================================================

print(f"Starting Model Training using {N_SPLITS}-Fold Cross-Validation...")

# Ensure parameters from Optuna are used, or fallback to default robust params
try:
    lgb_params = best_lgb_params
except NameError:
    print("Optuna params not found. Using fallback parameters...")
    lgb_params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_error',
        'boosting_type': 'gbdt',
        'class_weight': 'balanced', # Crucial for Imbalanced Dataset
        'random_state': RANDOM_STATE,
        'n_estimators': 600,
        'learning_rate': 0.05,
        'max_depth': 7,
        'num_leaves': 45
    }

fold_scores = []

# Iterate through the Stratified Folds defined in Step 3
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1}/{N_SPLITS} ---")
    
    # Split the data for current fold
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    # Initialize the model with balanced class weights
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    
    # Train the model with early stopping to prevent overfitting
    model_lgb.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Generate probability predictions for validation fold and test set
    # predict_proba returns probabilities for [Low(0), Medium(1), High(2)]
    val_preds_proba = model_lgb.predict_proba(X_val_fold)
    test_preds_proba = model_lgb.predict_proba(X_test)
    
    # Store OOF (Out-Of-Fold) probabilities for later blending
    oof_preds_lgb[val_idx] = val_preds_proba
    
    # Add fold's test predictions to the overall test predictions array
    # We divide by N_SPLITS to get the average probability across all folds
    test_preds_lgb += test_preds_proba / N_SPLITS
    
    # Convert probabilities to actual class predictions to calculate fold score
    val_preds_class = np.argmax(val_preds_proba, axis=1)
    fold_score = balanced_accuracy_score(y_val_fold, val_preds_class)
    fold_scores.append(fold_score)
    
    print(f"Fold {fold + 1} Balanced Accuracy: {fold_score:.5f}")

# Calculate Final OOF Score
# Convert accumulated OOF probabilities to class predictions
final_oof_class = np.argmax(oof_preds_lgb, axis=1)
overall_oof_score = balanced_accuracy_score(y, final_oof_class)

print("\n--- Step 5 Complete ---")
print(f"Average Fold Balanced Accuracy: {np.mean(fold_scores):.5f}")
print(f"Overall OOF Balanced Accuracy: {overall_oof_score:.5f}")

Starting Model Training using 5-Fold Cross-Validation...

--- Training Fold 1/5 ---
Fold 1 Balanced Accuracy: 0.96973

--- Training Fold 2/5 ---
Fold 2 Balanced Accuracy: 0.97162

--- Training Fold 3/5 ---
Fold 3 Balanced Accuracy: 0.97201

--- Training Fold 4/5 ---
Fold 4 Balanced Accuracy: 0.97072

--- Training Fold 5/5 ---
Fold 5 Balanced Accuracy: 0.97094

--- Step 5 Complete ---
Average Fold Balanced Accuracy: 0.97100
Overall OOF Balanced Accuracy: 0.97100


# Ensemble / Model Blending (Soft Voting)

In [6]:
import numpy as np

# =============================================================================
# STEP 6: ENSEMBLE / MODEL BLENDING (SOFT VOTING)
# =============================================================================

print("Starting Model Blending (Soft Voting)...")

# In a complete pipeline, you would have trained XGBoost and CatBoost similarly 
# to LightGBM in Step 5 and populated test_preds_xgb and test_preds_cat.
# We will check if they are populated. If they are still all zeros (untrained), 
# we will dynamically adjust the weights to rely only on the models that were trained.

is_xgb_trained = np.sum(test_preds_xgb) > 0
is_cat_trained = np.sum(test_preds_cat) > 0

# Define blending weights
weight_lgb = 1.0
weight_xgb = 0.0
weight_cat = 0.0

# Adjust weights based on which models are available
if is_xgb_trained and is_cat_trained:
    # If all 3 models are trained, use custom weights (can be tuned via Optuna too!)
    weight_lgb = 0.4
    weight_xgb = 0.3
    weight_cat = 0.3
elif is_xgb_trained:
    weight_lgb = 0.5
    weight_xgb = 0.5
elif is_cat_trained:
    weight_lgb = 0.5
    weight_cat = 0.5

print(f"Applied blending weights -> LGBM: {weight_lgb}, XGB: {weight_xgb}, Cat: {weight_cat}")

# Calculate the weighted average of probabilities (Soft Voting)
final_test_preds_proba = (
    (test_preds_lgb * weight_lgb) + 
    (test_preds_xgb * weight_xgb) + 
    (test_preds_cat * weight_cat)
)

# Convert the blended probabilities to final class predictions (0, 1, or 2)
# argmax returns the index of the highest probability
final_test_preds_class = np.argmax(final_test_preds_proba, axis=1)

print("\n--- Step 6 Complete ---")
print(f"First 5 blended probabilities:\n{final_test_preds_proba[:5]}")
print(f"First 5 final predicted classes (encoded): {final_test_preds_class[:5]}")

Starting Model Blending (Soft Voting)...
Applied blending weights -> LGBM: 1.0, XGB: 0.0, Cat: 0.0

--- Step 6 Complete ---
First 5 blended probabilities:
[[9.99986422e-01 1.35780277e-05 9.49054740e-11]
 [8.07007745e-01 1.92957624e-01 3.46305614e-05]
 [9.99981136e-01 1.88613608e-05 2.46025822e-09]
 [9.95740209e-01 4.25854089e-03 1.25012604e-06]
 [9.99925228e-01 7.47696821e-05 2.37949340e-09]]
First 5 final predicted classes (encoded): [0 0 0 0 0]


# Final Prediction and Submission Formulation

In [7]:
import pandas as pd

# =============================================================================
# STEP 7: FINAL PREDICTION AND SUBMISSION FORMULATION
# =============================================================================

print("Formulating final submission file...")

# Define the reverse mapping to convert numeric classes back to original text labels
reverse_target_mapping = {0: 'Low', 1: 'Medium', 2: 'High'}

# Convert the blended numeric predictions back to string categories
final_labels = [reverse_target_mapping[pred] for pred in final_test_preds_class]

# Create the final submission dataframe
# We use the 'id' column from the original test_df loaded in Step 1
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Irrigation_Need': final_labels
})

# Save the dataframe to a CSV file without the index column
submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False)

print("\n--- Step 7 Complete ---")
print(f"Successfully saved predictions to '{submission_filename}'")
print("\nPreview of the submission file:")
print(submission_df.head(10))

print("\nPredicted Class Distribution (Percentage):")
print(submission_df['Irrigation_Need'].value_counts(normalize=True) * 100)

print("\nPipeline finished! You can now upload 'submission.csv' to Kaggle. Good luck!")

Formulating final submission file...

--- Step 7 Complete ---
Successfully saved predictions to 'submission.csv'

Preview of the submission file:
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
5  630005          Medium
6  630006             Low
7  630007          Medium
8  630008            High
9  630009             Low

Predicted Class Distribution (Percentage):
Irrigation_Need
Low       59.155185
Medium    37.135556
High       3.709259
Name: proportion, dtype: float64

Pipeline finished! You can now upload 'submission.csv' to Kaggle. Good luck!
